# Weekly Claims ML Pipeline (Simple / No Pipelines)

This notebook:
1. Loads `data/weekly_model_dataset.parquet`
2. Applies leakage-safe feature filtering
3. Builds train/val/test time split
4. Trains two models (claim occurrence + claim type)
5. Saves validation/test predictions to parquet for external business validation


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    accuracy_score,
    log_loss,
    classification_report,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Prefer XGBoost when available
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

BASE_DIR = Path('..').resolve() / 'data'
DATA_PATH = BASE_DIR / 'weekly_model_dataset.parquet'
OUT_DIR = BASE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run prototype/feature_engineering.ipynb first.')

df = pd.read_parquet(DATA_PATH).copy()
df['week_start'] = pd.to_datetime(df['week_start'], errors='coerce')
df = df.dropna(subset=['resident_id', 'week_start']).sort_values(['resident_id', 'week_start']).reset_index(drop=True)

required = ['target_claim_next_week', 'target_claim_type_next_week']
for c in required:
    if c not in df.columns:
        raise ValueError(f'Missing required target: {c}')

print('shape:', df.shape)
print('date range:', df['week_start'].min(), '->', df['week_start'].max())
print('claim prevalence:', df['target_claim_next_week'].mean())


shape: (53985, 24)
date range: 2023-10-02 00:00:00 -> 2024-12-23 00:00:00
claim prevalence: 0.035343150875243125


## Time Split


In [3]:
weeks = np.array(sorted(df['week_start'].dropna().unique()))
if len(weeks) < 10:
    raise ValueError('Not enough weekly periods for train/val/test split.')

train_end = int(len(weeks) * 0.70)
val_end = int(len(weeks) * 0.85)

train_weeks = weeks[:train_end]
val_weeks = weeks[train_end:val_end]
test_weeks = weeks[val_end:]

df['split'] = np.where(
    df['week_start'].isin(train_weeks), 'train',
    np.where(df['week_start'].isin(val_weeks), 'val', 'test')
)

for s in ['train', 'val', 'test']:
    d = df[df['split'] == s]
    print(s, 'rows=', len(d), '|', d['week_start'].min(), '->', d['week_start'].max(), '| prevalence=', d['target_claim_next_week'].mean())


train rows= 34617 | 2023-10-02 00:00:00 -> 2024-08-05 00:00:00 | prevalence= 0.03405841060750498
val rows= 9268 | 2024-08-12 00:00:00 -> 2024-10-14 00:00:00 | prevalence= 0.03927492447129909
test rows= 10100 | 2024-10-21 00:00:00 -> 2024-12-23 00:00:00 | prevalence= 0.036138613861386136


## Leakage-Safe Feature Selection


In [4]:
# Explicit leakage and metadata columns to remove
explicit_drop = {
    'resident_id', 'week_start', 'week_end', 'split',
    'target_claim_next_week', 'target_claim_type_next_week',
    # known leakage / current-week claims-derived fields
    'claim_count', 'incident_type_primary', 'incident_type_list',
    'injury_count_sum', 'admission_count_sum', 'transfer_count_sum',
    'had_injury_week', 'had_admission_week', 'had_transfer_week', 'is_claim_week',
}

# Additional pattern-based guardrail
pattern_drop_prefixes = (
    'claim_', 'incident_type_', 'injury_count_', 'admission_count_', 'transfer_count_', 'had_', 'is_claim_'
)

leak_like_cols = [
    c for c in df.columns
    if c.startswith(pattern_drop_prefixes) and c not in {'target_claim_next_week', 'target_claim_type_next_week'}
]

drop_cols = explicit_drop.union(set(leak_like_cols))
feature_cols = [c for c in df.columns if c not in drop_cols]

print('total columns:', len(df.columns))
print('dropped columns:', len(drop_cols))
print('feature columns:', len(feature_cols))
print('sample features:', feature_cols[:15])


total columns: 25
dropped columns: 16
feature columns: 19
sample features: ['facility_id', 'resident_age', 'BP - Systolic_mean', 'BP - Systolic_std', 'Blood Sugar_mean', 'Blood Sugar_std', 'O2 sats_mean', 'O2 sats_std', 'Pain Level_mean', 'Pain Level_std', 'Pulse_mean', 'Pulse_std', 'Respiration_mean', 'Respiration_std', 'Temperature_mean']


## Direct Preprocessing Functions


In [5]:
train_mask = df['split'] == 'train'
val_mask = df['split'] == 'val'
test_mask = df['split'] == 'test'

X_train_raw = df.loc[train_mask, feature_cols].copy()
X_val_raw = df.loc[val_mask, feature_cols].copy()
X_test_raw = df.loc[test_mask, feature_cols].copy()

y_train_claim = df.loc[train_mask, 'target_claim_next_week'].astype(int).copy()
y_val_claim = df.loc[val_mask, 'target_claim_next_week'].astype(int).copy()
y_test_claim = df.loc[test_mask, 'target_claim_next_week'].astype(int).copy()

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X_train_raw[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

print('num cols:', len(num_cols), '| cat cols:', len(cat_cols))

# Fit preprocessing stats on training only (no leakage)
num_medians = X_train_raw[num_cols].median(numeric_only=True).to_dict()
cat_fill_value = '__MISSING__'


def transform_features(x_raw, num_cols, cat_cols, num_medians, train_dummy_cols=None):
    x = x_raw.copy()

    # Numeric
    for c in num_cols:
        x[c] = pd.to_numeric(x[c], errors='coerce').fillna(num_medians.get(c, 0.0))

    # Categorical
    for c in cat_cols:
        x[c] = x[c].astype(str).fillna(cat_fill_value)
        x[c] = x[c].replace({'nan': cat_fill_value, 'None': cat_fill_value, '': cat_fill_value})

    if len(cat_cols):
        x_cat = pd.get_dummies(x[cat_cols], prefix=cat_cols, dummy_na=False)
    else:
        x_cat = pd.DataFrame(index=x.index)

    x_num = x[num_cols].copy() if len(num_cols) else pd.DataFrame(index=x.index)
    x_out = pd.concat([x_num, x_cat], axis=1)

    if train_dummy_cols is None:
        train_dummy_cols = x_out.columns.tolist()

    # align to training schema
    x_out = x_out.reindex(columns=train_dummy_cols, fill_value=0)
    return x_out, train_dummy_cols


X_train, train_feature_matrix_cols = transform_features(X_train_raw, num_cols, cat_cols, num_medians)
X_val, _ = transform_features(X_val_raw, num_cols, cat_cols, num_medians, train_dummy_cols=train_feature_matrix_cols)
X_test, _ = transform_features(X_test_raw, num_cols, cat_cols, num_medians, train_dummy_cols=train_feature_matrix_cols)

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)


num cols: 18 | cat cols: 1
X_train shape: (34617, 118)
X_val shape: (9268, 118)
X_test shape: (10100, 118)


## Utility Metrics


In [6]:
def tune_threshold_f1(y_true, p, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 37)
    best_t, best_f1 = 0.5, -1
    for t in grid:
        pred = (p >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return float(best_t), float(best_f1)


def binary_metrics(y_true, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    return {
        'threshold': threshold,
        'prevalence': float(np.mean(y_true)),
        'roc_auc': roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc': average_precision_score(y_true, p),
        'f1': f1_score(y_true, pred, zero_division=0),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_true, pred),
        'mcc': matthews_corrcoef(y_true, pred) if len(np.unique(pred)) > 1 else 0.0,
        'accuracy': accuracy_score(y_true, pred),
    }


## Model 1: Claim Occurrence (Main + Baselines)


In [7]:
# Main binary model
if XGB_AVAILABLE:
    claim_main = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    main_name = 'xgb_main'
else:
    claim_main = RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    main_name = 'rf_main'

claim_main.fit(X_train, y_train_claim)

# Baselines
claim_dummy = DummyClassifier(strategy='prior', random_state=RANDOM_STATE)
claim_dummy.fit(X_train, y_train_claim)

claim_logit = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE)
claim_logit.fit(X_train, y_train_claim)

models_claim = {
    main_name: claim_main,
    'dummy_prior': claim_dummy,
    'logit_balanced': claim_logit,
}

# tune threshold on main model val predictions
p_val_main = claim_main.predict_proba(X_val)[:, 1]
best_t, best_f1 = tune_threshold_f1(y_val_claim.values, p_val_main)
print('best threshold for main model (val f1):', best_t, '| f1=', round(best_f1, 4))

rows = []
pred_cache_claim = {}
for name, mdl in models_claim.items():
    p_val = mdl.predict_proba(X_val)[:, 1]
    p_test = mdl.predict_proba(X_test)[:, 1]
    thr = best_t if name == main_name else 0.5

    m_val = binary_metrics(y_val_claim.values, p_val, threshold=thr)
    m_val.update({'split': 'val', 'model': name})
    rows.append(m_val)

    m_test = binary_metrics(y_test_claim.values, p_test, threshold=thr)
    m_test.update({'split': 'test', 'model': name})
    rows.append(m_test)

    pred_cache_claim[(name, 'val')] = (p_val, (p_val >= thr).astype(int), thr)
    pred_cache_claim[(name, 'test')] = (p_test, (p_test >= thr).astype(int), thr)

metrics_claim = pd.DataFrame(rows).sort_values(['split', 'pr_auc'], ascending=[True, False])
metrics_claim


best threshold for main model (val f1): 0.1 | f1= 0.204


,threshold,prevalence,roc_auc,pr_auc,f1,precision,recall,balanced_accuracy,mcc,accuracy,split,model
1,0.1,0.036139,0.728625,0.099887,0.153432,0.150794,0.156164,0.561595,0.121135,0.937723,test,xgb_main
5,0.5,0.036139,0.711999,0.077735,0.119112,0.066086,0.602740,0.641688,0.112511,0.677822,test,logit_balanced
3,0.5,0.036139,0.500000,0.036139,0.000000,0.000000,0.000000,0.500000,0.000000,0.963861,test,dummy_prior
0,0.1,0.039275,0.746173,0.125562,0.204030,0.188372,0.222527,0.591666,0.169305,0.931808,val,xgb_main
4,0.5,0.039275,0.727596,0.087271,0.133744,0.074455,0.656593,0.661462,0.131833,0.665947,val,logit_balanced
2,0.5,0.039275,0.500000,0.039275,0.000000,0.000000,0.000000,0.500000,0.000000,0.960725,val,dummy_prior


## Model 2: Claim Type (Conditional on Next-Week Claim)


In [8]:
# Type model data (only actual positive claim rows)
train_pos = train_mask & (df['target_claim_next_week'].astype(int) == 1)
val_pos = val_mask & (df['target_claim_next_week'].astype(int) == 1)
test_pos = test_mask & (df['target_claim_next_week'].astype(int) == 1)

X_train_type_raw = df.loc[train_pos, feature_cols].copy()
X_val_type_raw = df.loc[val_pos, feature_cols].copy()
X_test_type_raw = df.loc[test_pos, feature_cols].copy()

X_train_type, _ = transform_features(X_train_type_raw, num_cols, cat_cols, num_medians, train_dummy_cols=train_feature_matrix_cols)
X_val_type, _ = transform_features(X_val_type_raw, num_cols, cat_cols, num_medians, train_dummy_cols=train_feature_matrix_cols)
X_test_type, _ = transform_features(X_test_type_raw, num_cols, cat_cols, num_medians, train_dummy_cols=train_feature_matrix_cols)

y_train_type_raw = df.loc[train_pos, 'target_claim_type_next_week'].astype(str)
y_val_type_raw = df.loc[val_pos, 'target_claim_type_next_week'].astype(str)
y_test_type_raw = df.loc[test_pos, 'target_claim_type_next_week'].astype(str)

if len(y_train_type_raw) == 0:
    raise ValueError('No positive claim rows for type model training.')

le = LabelEncoder()
y_train_type = le.fit_transform(y_train_type_raw)
y_val_type = le.transform(y_val_type_raw) if len(y_val_type_raw) else np.array([], dtype=int)
y_test_type = le.transform(y_test_type_raw) if len(y_test_type_raw) else np.array([], dtype=int)

n_classes = len(le.classes_)
print('type classes:', n_classes, '|', list(le.classes_)[:10])

# Main type model
if XGB_AVAILABLE and n_classes >= 2:
    type_main = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='multi:softprob',
        num_class=n_classes,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    type_main_name = 'xgb_main'
else:
    type_main = RandomForestClassifier(
        n_estimators=600,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    type_main_name = 'rf_main'

type_main.fit(X_train_type, y_train_type)

type_dummy = DummyClassifier(strategy='prior', random_state=RANDOM_STATE)
type_dummy.fit(X_train_type, y_train_type)

type_logit = LogisticRegression(max_iter=3000, class_weight='balanced', solver='lbfgs', random_state=RANDOM_STATE)
type_logit.fit(X_train_type, y_train_type)

models_type = {
    type_main_name: type_main,
    'dummy_prior': type_dummy,
    'logit_balanced': type_logit,
}

rows = []
pred_cache_type = {}
for name, mdl in models_type.items():
    if len(y_val_type):
        p_val = mdl.predict_proba(X_val_type)
        pred_val = np.argmax(p_val, axis=1)
        rows.append({
            'split': 'val',
            'model': name,
            'accuracy': accuracy_score(y_val_type, pred_val),
            'balanced_accuracy': balanced_accuracy_score(y_val_type, pred_val),
            'f1_macro': f1_score(y_val_type, pred_val, average='macro', zero_division=0),
            'f1_weighted': f1_score(y_val_type, pred_val, average='weighted', zero_division=0),
            'log_loss': log_loss(y_val_type, p_val, labels=np.arange(n_classes)),
        })
        pred_cache_type[(name, 'val')] = (p_val, pred_val)

    if len(y_test_type):
        p_test = mdl.predict_proba(X_test_type)
        pred_test = np.argmax(p_test, axis=1)
        rows.append({
            'split': 'test',
            'model': name,
            'accuracy': accuracy_score(y_test_type, pred_test),
            'balanced_accuracy': balanced_accuracy_score(y_test_type, pred_test),
            'f1_macro': f1_score(y_test_type, pred_test, average='macro', zero_division=0),
            'f1_weighted': f1_score(y_test_type, pred_test, average='weighted', zero_division=0),
            'log_loss': log_loss(y_test_type, p_test, labels=np.arange(n_classes)),
        })
        pred_cache_type[(name, 'test')] = (p_test, pred_test)

metrics_type = pd.DataFrame(rows).sort_values(['split', 'f1_macro'], ascending=[True, False])
metrics_type


type classes: 6 | ['Altercation', 'Choking', 'Elopement', 'Fall', 'Medication Error', 'Wound']


/home/rubim/Documents/Projects/repos/tricura_case/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rubim/Documents/Projects/repos/tricura_case/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,split,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,log_loss
1,test,xgb_main,0.671233,0.286672,0.279095,0.608647,1.122598
3,test,dummy_prior,0.676712,0.250000,0.201797,0.546235,0.876925
5,test,logit_balanced,0.430137,0.297582,0.186357,0.488399,1.388215
0,val,xgb_main,0.706044,0.207319,0.212352,0.665616,0.849387
4,val,logit_balanced,0.442308,0.225750,0.196128,0.521215,1.439567
2,val,dummy_prior,0.741758,0.166667,0.141956,0.631781,0.759511


## Save Predictions for Validation and Test


In [9]:
# Choose the same main model names for export
claim_main_name = [k for k in models_claim.keys() if k not in {'dummy_prior', 'logit_balanced'}][0]
type_main_name = [k for k in models_type.keys() if k not in {'dummy_prior', 'logit_balanced'}][0]

# Binary predictions export
for split_name, mask, y_true in [
    ('val', val_mask, y_val_claim),
    ('test', test_mask, y_test_claim),
]:
    p, pred, thr = pred_cache_claim[(claim_main_name, split_name)]
    out = df.loc[mask, ['resident_id', 'week_start']].copy().reset_index(drop=True)
    out['y_claim_true'] = y_true.reset_index(drop=True)
    out['p_claim_pred'] = p
    out['y_claim_pred'] = pred
    out['threshold_used'] = thr

    out_path = OUT_DIR / f'{split_name}_claim_predictions.parquet'
    out.to_parquet(out_path, index=False)
    print('saved:', out_path, '| rows=', len(out))

# Type predictions export (on positive-claim rows only)
for split_name, pos_mask, y_true_raw in [
    ('val', val_pos, y_val_type_raw),
    ('test', test_pos, y_test_type_raw),
]:
    if (type_main_name, split_name) not in pred_cache_type:
        print('skip type export for', split_name, '(no positive rows)')
        continue

    p, pred_idx = pred_cache_type[(type_main_name, split_name)]
    pred_label = le.inverse_transform(pred_idx)

    out = df.loc[pos_mask, ['resident_id', 'week_start']].copy().reset_index(drop=True)
    out['y_type_true'] = y_true_raw.reset_index(drop=True).astype(str)
    out['y_type_pred'] = pd.Series(pred_label).astype(str)
    out['p_type_max'] = p.max(axis=1)

    # add per-class probabilities
    for i, cls in enumerate(le.classes_):
        out[f'p_type__{cls}'] = p[:, i]

    out_path = OUT_DIR / f'{split_name}_claim_type_predictions.parquet'
    out.to_parquet(out_path, index=False)
    print('saved:', out_path, '| rows=', len(out))


saved: /home/rubim/Documents/Projects/repos/tricura_case/data/val_claim_predictions.parquet | rows= 9268
saved: /home/rubim/Documents/Projects/repos/tricura_case/data/test_claim_predictions.parquet | rows= 10100
saved: /home/rubim/Documents/Projects/repos/tricura_case/data/val_claim_type_predictions.parquet | rows= 364
saved: /home/rubim/Documents/Projects/repos/tricura_case/data/test_claim_type_predictions.parquet | rows= 365


In [10]:
# Optional: save metrics summaries too
metrics_claim.to_parquet(OUT_DIR / 'metrics_claim.parquet', index=False)
metrics_type.to_parquet(OUT_DIR / 'metrics_claim_type.parquet', index=False)
print('saved metrics parquet files in', OUT_DIR)

display(metrics_claim)
display(metrics_type)


saved metrics parquet files in /home/rubim/Documents/Projects/repos/tricura_case/data


,threshold,prevalence,roc_auc,pr_auc,f1,precision,recall,balanced_accuracy,mcc,accuracy,split,model
1,0.1,0.036139,0.728625,0.099887,0.153432,0.150794,0.156164,0.561595,0.121135,0.937723,test,xgb_main
5,0.5,0.036139,0.711999,0.077735,0.119112,0.066086,0.602740,0.641688,0.112511,0.677822,test,logit_balanced
3,0.5,0.036139,0.500000,0.036139,0.000000,0.000000,0.000000,0.500000,0.000000,0.963861,test,dummy_prior
0,0.1,0.039275,0.746173,0.125562,0.204030,0.188372,0.222527,0.591666,0.169305,0.931808,val,xgb_main
4,0.5,0.039275,0.727596,0.087271,0.133744,0.074455,0.656593,0.661462,0.131833,0.665947,val,logit_balanced
2,0.5,0.039275,0.500000,0.039275,0.000000,0.000000,0.000000,0.500000,0.000000,0.960725,val,dummy_prior


,split,model,accuracy,balanced_accuracy,f1_macro,f1_weighted,log_loss
1,test,xgb_main,0.671233,0.286672,0.279095,0.608647,1.122598
3,test,dummy_prior,0.676712,0.250000,0.201797,0.546235,0.876925
5,test,logit_balanced,0.430137,0.297582,0.186357,0.488399,1.388215
0,val,xgb_main,0.706044,0.207319,0.212352,0.665616,0.849387
4,val,logit_balanced,0.442308,0.225750,0.196128,0.521215,1.439567
2,val,dummy_prior,0.741758,0.166667,0.141956,0.631781,0.759511
